In [ ]:
import torch
# Imports PyTorch.
# Provides tensors and the core deep-learning functionality.


import torch.nn as nn
# Imports PyTorch's neural-network layers and modules.
#
# "nn" gives us layers such as:
# Conv2d, BatchNorm2d, ReLU, MaxPool2d, Linear, Dropout, etc.


class FakeAudioCNN(nn.Module):
    # Defines our CNN model.
    #
    # nn.Module is the base class for PyTorch neural networks.
    #
    # Our model will learn to distinguish:
    #
    # 0 = REAL audio
    # 1 = FAKE audio
    #
    # The exact numerical labels depend on how the dataset
    # is created.


    def __init__(self):
        # Constructor for the neural network.
        #
        # This is where we define all of the layers.


        super().__init__()
        # Initializes the parent nn.Module class.
        #
        # This is required so PyTorch can properly track
        # the model's layers and trainable parameters.


        self.features = nn.Sequential(
            # "features" contains the convolutional part
            # of the network.
            #
            # Its job is to extract useful patterns from
            # the input audio representation.
            #
            # IMPORTANT:
            # Conv2d expects a 2D representation.
            #
            # For this project, the input would typically be
            # something like a spectrogram:
            #
            #          Time →
            #
            # Frequency
            #     ↓
            #
            #     [ spectrogram ]
            #
            # So the CNN learns patterns in the
            # time-frequency representation.


            # -------------------------
            # Block 1
            # -------------------------

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),
            # First convolutional layer.
            #
            # 1 input channel:
            # The spectrogram is represented as one channel.
            #
            # 32 output channels:
            # The network learns 32 different filters.
            #
            # kernel_size=3:
            # Each filter looks at a 3x3 region.
            #
            # padding=1:
            # Adds a one-pixel border around the input.
            #
            # This keeps the height and width approximately
            # the same after the convolution.
            #
            # Example:
            #
            # Input:
            # [1, H, W]
            #
            # Output:
            # [32, H, W]


            nn.BatchNorm2d(32),
            # Batch normalization for the 32 feature maps.
            #
            # Helps stabilize the values flowing through
            # the network during training.
            #
            # It can make training more stable and often
            # allows the model to converge faster.


            nn.ReLU(),
            # ReLU = Rectified Linear Unit.
            #
            # It introduces non-linearity into the network.
            #
            # ReLU(x) = max(0, x)
            #
            # Negative values become 0.
            # Positive values remain positive.


            nn.MaxPool2d(2),
            # Reduces the spatial dimensions by approximately
            # half.
            #
            # Example:
            #
            # [32, 128, 128]
            #
            # becomes:
            #
            # [32, 64, 64]
            #
            # This reduces computation and helps the network
            # focus on larger-scale patterns.


            # -------------------------
            # Block 2
            # -------------------------

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            # Second convolutional layer.
            #
            # Input channels = 32
            # Output channels = 64
            #
            # The network now learns 64 higher-level
            # feature representations.


            nn.BatchNorm2d(64),
            # Normalizes the 64 feature maps produced
            # by the convolution.


            nn.ReLU(),
            # Adds non-linearity after the convolution
            # and batch normalization.


            nn.MaxPool2d(2),
            # Again reduces height and width by approximately
            # half.
            #
            # Example:
            #
            # [64, 64, 64]
            #
            # becomes:
            #
            # [64, 32, 32]


            # -------------------------
            # Block 3
            # -------------------------

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),
            # Third convolutional layer.
            #
            # 64 input channels
            # 128 output channels
            #
            # The model can now represent increasingly
            # complex time-frequency patterns.


            nn.BatchNorm2d(128),
            # Normalizes the 128 feature maps.


            nn.ReLU(),
            # Applies the ReLU activation function.


            nn.MaxPool2d(2),
            # Reduces the time-frequency dimensions again.
            #
            # Example:
            #
            # [128, 32, 32]
            #
            # becomes:
            #
            # [128, 16, 16]


            # -------------------------
            # Block 4
            # -------------------------

            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                padding=1
            ),
            # Fourth convolutional layer.
            #
            # 128 input channels
            # 256 output channels
            #
            # This allows the network to learn more complex
            # audio patterns that may help distinguish
            # real speech from synthetic speech.


            nn.BatchNorm2d(256),
            # Normalizes the 256 feature maps.


            nn.ReLU(),
            # Applies another non-linear activation.


            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
            # Compresses every 256 feature map into
            # a single average value.
            #
            # Regardless of the original spatial dimensions,
            # the output becomes:
            #
            # [batch_size, 256, 1, 1]
            #
            # This is useful because we don't have to hard-code
            # the exact spectrogram height/width before the
            # classifier.


        )


        self.classifier = nn.Sequential(
            # This section takes the features extracted
            # by the CNN and performs the final classification.


            nn.Flatten(),
            # Converts:
            #
            # [batch_size, 256, 1, 1]
            #
            # into:
            #
            # [batch_size, 256]
            #
            # This prepares the data for the Linear layer.


            nn.Linear(
                256,
                128
            ),
            # Fully connected layer.
            #
            # 256 input features
            # 128 output features
            #
            # Combines the extracted CNN features into
            # a more compact representation.


            nn.ReLU(),
            # Adds non-linearity to the classifier.


            nn.Dropout(0.3),
            # Randomly turns off approximately 30% of neurons
            # during training.
            #
            # This helps reduce overfitting.
            #
            # During evaluation, Dropout is automatically
            # disabled when model.eval() is used.


            nn.Linear(
                128,
                2
            )
            # Final classification layer.
            #
            # 128 input features
            # 2 output values
            #
            # The two outputs represent the two classes:
            #
            # REAL
            # FAKE
            #
            # These outputs are called logits.
            #
            # CrossEntropyLoss can take these logits directly;
            # we do NOT need to apply softmax inside the model.


        )


    def forward(self, x):
        # Defines how input data flows through the network.
        #
        # x is the input spectrogram tensor.
        #
        # Expected general shape:
        #
        # [batch_size, 1, frequency, time]


        x = self.features(x)
        # Sends the input through all convolutional layers.
        #
        # The CNN extracts time-frequency features
        # from the audio representation.


        x = self.classifier(x)
        # Sends the extracted features through the
        # fully connected classification layers.
        #
        # Produces two logits:
        #
        # [real_score, fake_score]


        return x
        # Returns the final classification logits.

